# Entrega 1 — Análise Comparativa de Representações
**Ouvidoria Inteligente: Triagem Semântica de Manifestações Cidadãs**

Comparação entre representações esparsas (Bag-of-Words, TF-IDF) e densas (Embeddings) sobre o corpus de 40 manifestações da ouvidoria, incluindo o cálculo de similaridade de cosseno para os pares indicados no enunciado.

## 1. Carregamento do corpus

In [ ]:
import json
import numpy as np
import pandas as pd

with open("manifestacoes.json", encoding="utf-8") as f:
    manifestacoes = json.load(f)

df = pd.DataFrame(manifestacoes)
textos = df["texto"].tolist()
ids = df["id"].tolist()
id_to_idx = {i: idx for idx, i in enumerate(ids)}

print(f"{len(df)} manifestações carregadas.")
df.head()

## 2. Representação Esparsa: Bag-of-Words (BoW)
BoW conta a frequência bruta de cada termo, sem qualquer noção de importância relativa ou de significado.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow = CountVectorizer()
X_bow = bow.fit_transform(textos)
print(f"Vocabulário BoW: {len(bow.vocabulary_)} termos")
print(f"Matriz BoW: {X_bow.shape} (manifestações × termos)")

## 3. Representação Esparsa: TF-IDF
TF-IDF pondera os termos pela sua raridade na coleção, reduzindo o peso de palavras muito comuns.

In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(textos)
print(f"Matriz TF-IDF: {X_tfidf.shape}")

## 4. Representação Densa: Embeddings
Usamos um modelo multilíngue de *sentence embeddings*, capaz de capturar o significado da frase (não apenas as palavras que ela contém).

In [ ]:
from sentence_transformers import SentenceTransformer

NOME_MODELO = "paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOME_MODELO)
X_emb = modelo.encode(textos, show_progress_bar=True, normalize_embeddings=True)
print(f"Matriz de Embeddings ({NOME_MODELO}): {X_emb.shape}")

## 5. Comparação de Similaridade de Cosseno — Pares do Enunciado
- **M003 × M017**: "buraco enorme na Av. Brasil" × "asfalto todo esburacado da avenida principal" — mesma reclamação, vocabulário diferente.
- **M008 × M022**: "posto de saúde sem médico" × "falta atendimento no PSF" — mesmo problema (falta de médico), termos técnicos diferentes (PSF).
- **M008 × M031**: "posto de saúde sem médico" × "lâmpada queimada na praça" — problemas completamente distintos (par de controle).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def cosseno(matriz, i, j):
    """Similaridade de cosseno entre as linhas i e j de uma matriz esparsa ou densa."""
    vi = matriz[i]
    vj = matriz[j]
    if hasattr(vi, "toarray"):
        vi, vj = vi.toarray(), vj.toarray()
    else:
        vi, vj = vi.reshape(1, -1), vj.reshape(1, -1)
    return float(cosine_similarity(vi, vj)[0][0])

pares = [("M003", "M017"), ("M008", "M022"), ("M008", "M031")]

linhas = []
for a, b in pares:
    ia, ib = id_to_idx[a], id_to_idx[b]
    linhas.append({
        "Par": f"{a} × {b}",
        "BoW": round(cosseno(X_bow, ia, ib), 4),
        "TF-IDF": round(cosseno(X_tfidf, ia, ib), 4),
        "Embeddings": round(cosseno(X_emb, ia, ib), 4),
    })

df_comparacao = pd.DataFrame(linhas)
df_comparacao

### Textos completos dos pares (para referência na discussão)

In [ ]:
for a, b in pares:
    print(f"--- {a} × {b} ---")
    print(f"{a}: {df.loc[id_to_idx[a], 'texto']}")
    print(f"{b}: {df.loc[id_to_idx[b], 'texto']}")
    print()

## 6. Discussão dos Resultados

**M003 × M017** (mesmo problema, vocabulário diferente — "buraco" vs. "esburacado"/"avenida principal"): espera-se BoW e TF-IDF baixos, pois compartilham poucas palavras exatas, enquanto os Embeddings devem apontar similaridade alta por capturarem que as duas frases descrevem o mesmo problema de pavimentação.

**M008 × M022** (mesmo problema, com um termo técnico do domínio — "PSF"): situação semelhante — a sobreposição lexical é baixa ("posto de saúde" vs. "PSF", "sem médico" vs. "falta atendimento"), mas o significado é o mesmo. Os Embeddings, por terem sido treinados também com contexto de saúde pública, tendem a aproximar os dois vetores.

**M008 × M031** (par de controle, temas não relacionados): aqui os três métodos devem convergir para uma similaridade baixa, já que não há nem sobreposição lexical nem semântica relevante entre "falta de médico" e "lâmpada queimada".

### Limitações de cada abordagem

- **Bag-of-Words**: ignora completamente a ordem das palavras e não possui noção de sinônimos ou de importância relativa dos termos — duas frases sobre o mesmo assunto, mas escritas com vocabulário diferente, são tratadas como quase ortogonais no espaço vetorial.
- **TF-IDF**: resolve parcialmente o problema de termos muito frequentes (como "o", "de", "que"), mas continua sendo uma representação esparsa e lexical — ainda não reconhece que "buraco" e "esburacado" ou "médico" e "atendimento" se referem ao mesmo problema.
- **Embeddings**: capturam relações semânticas e sinônimos, sendo a abordagem que melhor identifica duplicatas reais na ouvidoria. Em contrapartida, é computacionalmente mais custosa, exige um modelo pré-treinado (dependência externa) e pode, em casos raros, aproximar textos que soam parecidos mas tratam de assuntos distintos (falsos positivos), algo que TF-IDF dificilmente faria para termos muito específicos (como códigos ou nomes próprios).